In [1]:
# %%
# STEP 1: Setup — import libraries and define helper function

import subprocess
import os

def run(cmd):
    print(f"\n>>> {cmd}")
    subprocess.run(cmd, shell=True, executable="/bin/bash")

In [2]:
# %%
# STEP 2: Define root directory and simulator path

base_dir = "/Users/jakegehrung/Desktop/data_raw/baseline_models"
run_simulator_path = "/Users/jakegehrung/Desktop/data_raw/ESP-r_automation_txt/run_simulator.txt"

print("Base directory:", base_dir)

Base directory: /Users/jakegehrung/Desktop/data_raw/baseline_models


In [3]:
# %%
# STEP 3: Load environment — ensure ESP-r tools are available

os.chdir(os.path.expanduser("~"))

run("source .profile")
run("which bps")  # check ESP-r simulator is available


>>> source .profile

>>> which bps


In [4]:
# %%
# STEP 4: Discover all model directories — loop through buildings and valid model folders

valid_model_names = {
    "SemiDetached_2F_walls","Detached_2F_walls"
}

model_paths = []

for building in os.listdir(base_dir):
    building_path = os.path.join(base_dir, building)
    
    if not os.path.isdir(building_path):
        continue
    
    for sub in os.listdir(building_path):
        sub_path = os.path.join(building_path, sub)
        
        if sub in valid_model_names and os.path.isdir(sub_path):
            model_paths.append(sub_path)

print(f"Found {len(model_paths)} model(s)")

Found 70 model(s)


In [5]:
# %%
# STEP 5: Loop through each model — clean, run simulation, verify output

for model_root in model_paths:
    
    print("\n" + "="*60)
    print(f"Processing model: {model_root}")
    
    cfg_dir = os.path.join(model_root, "cfg")
    tmp_dir = os.path.join(model_root, "tmp")
    
    model_name = os.path.basename(model_root)
    cfg_file = f"SemiDetached_2F.cfg"
    
    print("CFG dir:", cfg_dir)
    print("TMP dir:", tmp_dir)
    
    # --- Clean tmp directory ---
    if os.path.exists(tmp_dir):
        os.chdir(tmp_dir)
        run("ls")
        
        files_to_remove = [
            "annual.res",
            f"{model_name}.res",
            "SemiDetached_2F.res",
            "AH_RESULTS.res"
        ]
        
        for f in files_to_remove:
            if os.path.exists(f):
                print(f"Removing {f}")
                os.remove(f)
            else:
                print(f"{f} not found (skipping)")
    else:
        print("WARNING: tmp directory not found, skipping model")
        continue
    
    # --- Run simulation ---
    if os.path.exists(cfg_dir):
        os.chdir(cfg_dir)
        run("ls")
        
        cmd = f"""
        source ~/.profile
        bps -mode script -file {cfg_file} < {run_simulator_path}
        """
        
        run(cmd)
    else:
        print("WARNING: cfg directory not found, skipping model")
        continue
    
    # --- Verify output ---
    os.chdir(tmp_dir)
    run("ls")
    
    if os.path.exists("annual.res"):
        print("Simulation successful: annual.res created")
    else:
        print("WARNING: annual.res not found — check simulation logs")


Processing model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls
CFG dir: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls/cfg
TMP dir: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls/tmp

>>> ls
annual.res
Removing annual.res
SemiDetached_2F_walls.res not found (skipping)
SemiDetached_2F.res not found (skipping)
AH_RESULTS.res not found (skipping)

>>> ls
ACH.txt
SemiDetached_2F.cfg
SemiDetached_2F.cnn
annual.res
diffuse_horizonal_solar_raw.csv
direct_normal_solar_raw.csv
fort.27
fort.942
graphic.err
graphic.out
heat_load.csv
heat_load_raw.csv
latitude.txt
longitude.txt
pv
pv_power.csv
pv_res.csv
roof-2_surface_LW.csv
roof.txt
rotate.txt
update_cons.txt
walls.txt
windows.txt

>>> 
        source ~/.profile
        bps -mode script -file SemiDetached_2F.cfg < /Users/jakegehrung/Desktop/data_raw/ESP-r_automation_txt/run_simulator.txt
        
 
Welcome to the Integrated simula

 
Integrated simulator:
  a define model                w warnings >> OFF          
  b assign weather file         r reporting >> silent      
    _________________________     _______________________  
  c initiate simulation         s H3K reports unavailable  
    _________________________     _________________________
  t trace facilities            ? help                     
  y multi-year sim >> OFF       - quit module              

 Integrated simulator:?> (currently: BLDres)
 
Building results file name? 

(currently:  9  1)
Simulation period
Start day & month? (currently: 15  1)
Simulation period
End day & month? (currently: 1)
 
Number of start-up days? (currently: 1)
 
Building time-steps/hour?  
Hourly results integration? [Y]es [N]o ?
Simulation control:
  a results library                                          
  b simulation period            < delete last result set    
    ....................           ....................      
  * Save >> 2 (1+ surf temps)    g simulation options        
    ....................         i view simulation parameters
  m monitor progress             o view save level 0 result  
  s commence simulation            ....................      
  t time-step control            ? help                      
  - exit menu

 Simulation control:?>  Use ../ctl/SemiDetached_2F.ctl
 with the current model? [Y]es [N]o ?
 
 Within the current model  2 control loops have been specified.
 The overall project control is
heating system control
 and the zone con

 
Model configuration file: SemiDetached_2F.cfg
Model description: Semi-Detached, 2-floor archetype
Weather file: ../dbs/fintryclimate.epw
Control file: ../ctl/SemiDetached_2F.ctl
Building results file: ../tmp/annual.res
Save option: 2 (Moderate)
Prior warnings:      0
 
Simulation period: 01-Jan  @ 01h00 to 31-Dec  @ 24h00
Start-up period:      3 day(s).
Building time steps: 30.00 minutes.
Number of zones:      3.
Building results file size:     24.1 Mbytes.
 
Simulation commenced.


CPU time:      0.5s.
 
Save results? [Y]es [N]o ?
 
Saved building result-set  1
   for period day  1 month  1 to day 31 month 12
Simulation control:
  a results library                                          
  b simulation period            < delete last result set    
    ....................           ....................      
  * Save >> 2 (1+ surf temps)    g simulation options        
    ....................         i view simulation parameters
  m monitor progress             o view save level 0 result  
  s commence simulation            ....................      
  t time-step control            ? help                      
  - exit menu

 Simulation control:?>  
Integrated simulator:
  a define model                w warnings >> OFF          
  b assign weather file         r reporting >> silent      
    _________________________     _______________________  
  c initiate simulation         s H3K reports unavailable  
    _________________________     __________________

...

 
Model configuration file: SemiDetached_2F.cfg
Model description: Semi-Detached, 2-floor archetype
Weather file: ../dbs/fintryclimate.epw
Control file: ../ctl/SemiDetached_2F.ctl
Building results file: ../tmp/annual.res
Save option: 2 (Moderate)
Prior warnings:      0
 
Simulation period: 01-Jan  @ 01h00 to 31-Dec  @ 24h00
Start-up period:      3 day(s).
Building time steps: 30.00 minutes.
Number of zones:      3.
Building results file size:     24.1 Mbytes.
 
Simulation commenced.


CPU time:      0.5s.
 
Save results? [Y]es [N]o ?
 
Saved building result-set  1
   for period day  1 month  1 to day 31 month 12
Simulation control:
  a results library                                          
  b simulation period            < delete last result set    
    ....................           ....................      
  * Save >> 2 (1+ surf temps)    g simulation options        
    ....................         i view simulation parameters
  m monitor progress             o view save level 0 result  
  s commence simulation            ....................      
  t time-step control            ? help                      
  - exit menu

 Simulation control:?>  
Integrated simulator:
  a define model                w warnings >> OFF          
  b assign weather file         r reporting >> silent      
    _________________________     _______________________  
  c initiate simulation         s H3K reports unavailable  
    _________________________     __________________

(currently:  9  1)
Simulation period
Start day & month? (currently: 15  1)
Simulation period
End day & month? (currently: 15)
 
Number of start-up days? (currently: 1)
 
Building time-steps/hour?  
Hourly results integration? [Y]es [N]o ?
Simulation control:
  a results library                                          
  b simulation period            < delete last result set    
    ....................           ....................      
  * Save >> 2 (1+ surf temps)    g simulation options        
    ....................         i view simulation parameters
  m monitor progress             o view save level 0 result  
  s commence simulation            ....................      
  t time-step control            ? help                      
  - exit menu

 Simulation control:?>  Use ../ctl/SemiDetached_2F.ctl
 with the current model? [Y]es [N]o ?
 
 Within the current model  2 control loops have been specified.
 The overall project control is
heating system control
 and the zone co

 
Model configuration file: SemiDetached_2F.cfg
Model description: Semi-Detached, 2-floor archetype
Weather file: ../dbs/fintryclimate.epw
Control file: ../ctl/SemiDetached_2F.ctl
Building results file: ../tmp/annual.res
Save option: 2 (Moderate)
Prior warnings:      0
 
Simulation period: 01-Jan  @ 01h00 to 31-Dec  @ 24h00
Start-up period:      3 day(s).
Building time steps: 30.00 minutes.
Number of zones:      3.
Building results file size:     24.1 Mbytes.
 
Simulation commenced.


CPU time:      0.5s.
 
Save results? [Y]es [N]o ?
 
Saved building result-set  1
   for period day  1 month  1 to day 31 month 12
Simulation control:
  a results library                                          
  b simulation period            < delete last result set    
    ....................           ....................      
  * Save >> 2 (1+ surf temps)    g simulation options        
    ....................         i view simulation parameters
  m monitor progress             o view save level 0 result  
  s commence simulation            ....................      
  t time-step control            ? help                      
  - exit menu

 Simulation control:?>  
Integrated simulator:
  a define model                w warnings >> OFF          
  b assign weather file         r reporting >> silent      
    _________________________     _______________________  
  c initiate simulation         s H3K reports unavailable  
    _________________________     __________________

In [6]:
# %%
# STEP 6: Copy results — move annual.res from tmp to cfg (overwrite if exists)

import shutil

for model_root in model_paths:
    
    print("\n" + "="*60)
    print(f"Copying results for model: {model_root}")
    
    cfg_dir = os.path.join(model_root, "cfg")
    tmp_dir = os.path.join(model_root, "tmp")
    
    src_file = os.path.join(tmp_dir, "annual.res")
    dst_file = os.path.join(cfg_dir, "annual.res")
    
    if not os.path.exists(src_file):
        print("WARNING: annual.res not found in tmp — skipping")
        continue
    
    if not os.path.exists(cfg_dir):
        print("WARNING: cfg directory not found — skipping")
        continue
    
    # Copy and overwrite if exists
    try:
        shutil.copy2(src_file, dst_file)
        print(f"Copied annual.res to {dst_file} (overwritten if existed)")
    except Exception as e:
        print(f"ERROR copying file: {e}")


Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664924/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829085/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063637/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001063637/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664941/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664941/Detached_2F_walls/cfg/annual.res (over

Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870876/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870882/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870882/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951902/Detached_2F_walls


Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951902/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234621926/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1234621926/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627539/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627539/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627541/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627541/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1236594950/Detach

Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627549/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627540/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627540/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627547/Detached_2F_walls


Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627547/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870854/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870854/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1235812262/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1235812262/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1000336709/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1000336709/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/100023891

...

Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627567/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234647968/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1234647968/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627558/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627558/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1002143352/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1002143352/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001627542/Detached_2F_wa

Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664932/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829058/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829058/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664935/SemiDetached_2F_walls


Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664935/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951858/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1234760996/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1234760996/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870859/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001870859/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829059/SemiDetach

Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664934/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829061/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829061/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238911/SemiDetached_2F_walls


Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238911/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829068/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829068/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238920/SemiDetached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1000238920/SemiDetached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951857/Detached_2F_walls
Copied annual.res to /Users/jakegehrung/Desktop/data_raw/baseline_models/1001951857/Detached_2F_walls/cfg/annual.res (overwritten if existed)

Copying results for model: /Users/jakegehrung/Desktop/data_raw/baseline_models/10010